In [1]:
import copy

import pandas as pd

from hypex.dataset import Dataset, ExperimentData
from hypex.dataset.roles import FeatureRole, InfoRole, TargetRole

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType
import pyspark.pandas as ps
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import tempfile
import os

from hypex.dataset.backends import SparkDataset

# Создаем Spark сессию
sp_s = (SparkSession.builder 
    .appName("HypEx-Spark-Tests") 
    .master("local[*]") 
    .config("spark.driver.memory", "4g") 
    .config("spark.sql.shuffle.partitions", "4") 
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

sp_s.sparkContext.setLogLevel("WARN")
print(f"Spark Version: {sp_s.version}")
from hypex.utils import BackendsEnum

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(
26/03/05 14:36:06 WARN Utils: Your hostname, MacBook-Pro-Danil.local resolves to a loopback address: 127.0.0.1; using 10.240.120.240 instead (on interface en0)
26/03/05 14:36:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/05 14:36:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Version: 3.5.1


26/03/05 14:36:07 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
df = pd.DataFrame({'a': [1, 2, 3], 'b': [4, 5, 6]})

ds = Dataset({'a': TargetRole(), 'b': TargetRole(float)}, 
             data=df,
             backend=BackendsEnum.spark,
             session=sp_s)
ds["a"]

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/

,a
0,1
1,2
2,3


In [3]:
ds

,a,b
0,1,4.0
1,2,5.0
2,3,6.0


In [4]:
columns_found = ds.search_columns(TargetRole(), search_types=[int])
columns_found

['a']

In [5]:
ds.select('a')

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


,a
0,1
1,2
2,3


In [6]:
ds.count()

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


,a,b
count,3,3


In [7]:
ds.log()

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_numpy` loads all data into the driver's memory. It should only be used if the resulting NumPy ndarray is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


,a,b
0,0.000000,1.386294
1,0.693147,1.609438
2,1.098612,1.791759


In [8]:
ds.min()

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


,a,b
min,1,4


In [9]:
ds[1]

,1
a,2.0
b,5.0


In [10]:
ds['a'][1]

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


,1
a,2


In [11]:
ds[ds[['a', 'b']] == 4]

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/

,a,b
0,1,4.0
1,2,5.0
2,3,6.0


In [12]:
ds['c'] = [-3, -7, -9]

/Users/danilsamsutdinov/HypEx/hypex/dataset/abstract.py:383: SyntaxWarning: Column must be added by using add_column method.
  warnings.warn(
26/03/05 14:36:14 WARN AttachDistributedSequenceExec: clean up cached RDD(176) in AttachDistributedSequenceExec(1046)


In [13]:
ds

26/03/05 14:36:14 WARN AttachDistributedSequenceExec: clean up cached RDD(201) in AttachDistributedSequenceExec(1322)
26/03/05 14:36:14 WARN AttachDistributedSequenceExec: clean up cached RDD(197) in AttachDistributedSequenceExec(1318)


,a,b,c
0,1,4.0,-3
1,2,5.0,-7
2,3,6.0,-9


In [14]:
ds.add_column([7, 8, 9], {'d': TargetRole(int)})

26/03/05 14:36:15 WARN AttachDistributedSequenceExec: clean up cached RDD(221) in AttachDistributedSequenceExec(1758)
26/03/05 14:36:15 WARN AttachDistributedSequenceExec: clean up cached RDD(247) in AttachDistributedSequenceExec(2282)
26/03/05 14:36:15 WARN AttachDistributedSequenceExec: clean up cached RDD(282) in AttachDistributedSequenceExec(2855)
26/03/05 14:36:15 WARN AttachDistributedSequenceExec: clean up cached RDD(278) in AttachDistributedSequenceExec(2851)


,a,b,c,d
2,3,6.0,-9,9
0,1,4.0,-3,7
1,2,5.0,-7,8


In [15]:
ds.apply(lambda x: x ** 2 , role={'a': TargetRole(int), 'b': TargetRole(float), 'c': TargetRole(float)})

26/03/05 14:36:16 WARN AttachDistributedSequenceExec: clean up cached RDD(312) in AttachDistributedSequenceExec(3609)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If the type hints is not specified for `apply`, it is expensive to infer the data type internally.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
26/03/05 14:36:16 WARN AttachDistributedSequenceExec: clean up cached RDD(346) in AttachDistributedSequenceExec(4317)
26/03/05 14:36:16 WARN AttachDistributedSequenceExec: clean up cached RDD(342) in AttachDistributedSequenceExec(4313)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark

,a,b,c,d
1,4,25.0,49.0,81
2,9,36.0,81.0,49
0,1,16.0,9.0,64


In [16]:
ds

,a,b,c,d
2,3,6.0,-9,9
0,1,4.0,-3,7
1,2,5.0,-7,8


In [17]:
ds.groupby('a').sum("b")

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
26/03/05 14:36:17 WARN AttachDistributedSequenceExec: clean up cached RDD(393) in AttachDistributedSequenceExec(5212)
26/03/05 14:36:17 WARN AttachDistributedSequenceExec: clean up cached RDD(426) in AttachDistributedSequenceExec(6193)


,b
a,
2,5.0
1,4.0
3,6.0


In [18]:
groups = ds.groupby('a')
groups

In [ ]:
ds.transpose({'one': FeatureRole(), '2': InfoRole(), 'III': InfoRole()})

In [ ]:
ds.sample(frac=1.0)

In [ ]:
ds.append(ds)

In [ ]:
ed = ExperimentData(ds)